# NLP Pipeline

importing libraries 

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns   
import os 


In [41]:
for dirname, _, filenames in os.walk('../data/raw'):
    for filename in filenames:
        print(os.path.join(dirname, filename))  

../data/raw\db.txt
../data/raw\train.txt


read the file as a dataframe and convert it to csv style

In [42]:
df = pd.read_csv('../data/raw/train.txt', sep = ';', names = ['text', 'emotion'])
df.head()           

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [43]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [44]:
uniqe = df.emotion.unique()
uniqe

<ArrowStringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

Emotion score for each emotion because machine learning is all about numbers

In [45]:
emotion_no = {}
i = 0
for em in uniqe:
    emotion_no[em] = i
    i = i+1
print(emotion_no)   


{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [46]:
df['emotion'] = df['emotion'].map(emotion_no)                   
df.head()           

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Text cleaning

converting to lower case

lambda function use to lowercase

In [47]:
df['text'] = df['text'].apply(lambda x: x.lower())
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


Remove special characters   

In [48]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)

## Remove Numbers

In [49]:
def remove_number(text):
    return text.translate(str.maketrans('', '', string.digits))

df['text'] = df['text'].apply(remove_number)
df.head()                       

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Remove URLS

In [50]:
import re

def remove_url(text):
        # Regex matches http/https URLs and www. links
    return re.sub(r'http\S+|www\S+', '', text)

df['text'] = df['text'].apply(remove_url)                   
df.head()   

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Removes Emojies

using ASCII method

In [51]:
def emoji_remove(text):
    new = ""
    for i in text:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(emoji_remove)                   
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


using emoji library 

In [52]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.


In [53]:
import emoji

def remove_emojies(text):
    return ''.join(char for char in text if char not in emoji.EMOJI_DATA)

df['text'] = df['text'].apply(remove_emojies)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Removing Stop Words

In [54]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\Manab
[nltk_data]     Biswas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Manab
[nltk_data]     Biswas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Manab
[nltk_data]     Biswas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [55]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [56]:
def remove_stopwords(txt):
    word = word_tokenize(txt)
    cleaned = []
    for i in word:
        if i not in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)



In [57]:
df['text'] = df['text'].apply(remove_stopwords)

df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


### Feature extraction / Vectorization

1. One Hot Encoding

2. Bag of Words

3. TF-IDF